# Text Classification with Zero-Shot Learning

This notebook demonstrates text classification using zero-shot learning, which allows us to classify text into categories without training data!

## What is Zero-Shot Classification?

Zero-shot classification is a technique where a model can classify text into categories it hasn't been explicitly trained on. This is incredibly powerful for dynamic categorization tasks.

## Setup

In [ ]:
from transformers import pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("Libraries imported successfully!")

## Load Zero-Shot Classification Model

We'll use BART fine-tuned on MNLI (Multi-Genre Natural Language Inference) dataset.

In [ ]:
# Load the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", 
                     model="facebook/bart-large-mnli")

print("Model loaded successfully!")

## Single Text Classification

Let's classify a single text into multiple categories:

In [ ]:
# Example text
text = "Apple announced the new iPhone 15 with improved camera features and longer battery life."

# Define possible categories
categories = ["technology", "sports", "politics", "entertainment", "science"]

# Classify
result = classifier(text, categories)

print(f"Text: {text}\n")
print(f"Top prediction: {result['labels'][0]} (confidence: {result['scores'][0]:.4f})\n")
print("All predictions:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:15s}: {score:.4f}")

## Visualize Single Classification

In [ ]:
# Create visualization
plt.figure(figsize=(10, 6))
plt.barh(result['labels'], result['scores'], color='steelblue')
plt.xlabel('Confidence Score', fontsize=12)
plt.ylabel('Category', fontsize=12)
plt.title('Classification Confidence Scores', fontsize=14, fontweight='bold')
plt.xlim(0, 1)
for i, (label, score) in enumerate(zip(result['labels'], result['scores'])):
    plt.text(score + 0.01, i, f'{score:.3f}', va='center')
plt.tight_layout()
plt.show()

## Batch Classification

Let's classify multiple texts:

In [ ]:
# Multiple example texts
texts = [
    "Apple announced the new iPhone 15 with improved camera features.",
    "The Lakers won the championship game with a score of 102-98.",
    "The Senate passed a new bill regarding climate change policy.",
    "The latest Marvel movie broke box office records worldwide.",
    "Researchers discovered a new species of deep-sea fish.",
    "Google released an update to its search algorithm.",
]

# Classify all texts
results = []
for text in texts:
    result = classifier(text, categories)
    results.append({
        'text': text[:50] + '...' if len(text) > 50 else text,
        'category': result['labels'][0],
        'confidence': result['scores'][0]
    })

# Create DataFrame
df = pd.DataFrame(results)
print(df.to_string(index=False))

## Visualize Batch Results

In [ ]:
# Category distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Category counts
category_counts = df['category'].value_counts()
category_counts.plot(kind='bar', ax=ax1, color='coral')
ax1.set_title('Category Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Category', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# Confidence distribution
df.groupby('category')['confidence'].mean().plot(kind='bar', ax=ax2, color='steelblue')
ax2.set_title('Average Confidence by Category', fontsize=14, fontweight='bold')
ax2.set_xlabel('Category', fontsize=12)
ax2.set_ylabel('Average Confidence', fontsize=12)
ax2.set_ylim(0, 1)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Multi-Label Classification

Sometimes text can belong to multiple categories. Let's explore that:

In [ ]:
# Text that could fit multiple categories
multi_text = "The tech CEO announced a major sponsorship deal with a sports team valued at $100 million."

result = classifier(multi_text, categories, multi_label=True)

print(f"Text: {multi_text}\n")
print("Relevance scores for each category:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:15s}: {score:.4f}")

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(result['labels'], result['scores'], color='teal')
plt.xlabel('Relevance Score', fontsize=12)
plt.ylabel('Category', fontsize=12)
plt.title('Multi-Label Classification Scores', fontsize=14, fontweight='bold')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## Interactive: Classify Your Own Text!

Try classifying your own text with custom categories:

In [ ]:
# Customize these!
your_text = "The new restaurant downtown serves amazing Italian cuisine."
your_categories = ["food", "travel", "health", "business", "education"]

# Classify
result = classifier(your_text, your_categories)

print(f"Your text: {your_text}\n")
print(f"Top prediction: {result['labels'][0]} (confidence: {result['scores'][0]:.4f})\n")

# Visualize
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(result['scores'])
plt.barh(result['labels'], result['scores'], color=colors)
plt.xlabel('Confidence Score', fontsize=12)
plt.ylabel('Category', fontsize=12)
plt.title('Your Text Classification', fontsize=14, fontweight='bold')
plt.xlim(0, 1)
for i, (label, score) in enumerate(zip(result['labels'], result['scores'])):
    plt.text(score + 0.01, i, f'{score:.3f}', va='center')
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Zero-shot classification** enables flexible categorization without training data
2. **Dynamic categories**: You can change categories on the fly
3. **Multi-label support**: Text can belong to multiple categories
4. **Confidence scores** help assess prediction reliability
5. **Real-world applications**:
   - Content moderation
   - Document organization
   - Email routing
   - News categorization
   - Customer support ticket classification

## Limitations

- Slower than traditional ML for fixed categories
- May struggle with very domain-specific categories
- Requires clear, distinct category names

## Next Steps

- Compare with traditional ML (TF-IDF + Logistic Regression)
- Fine-tune on domain-specific data for better accuracy
- Implement hierarchical classification
- Build a classification API service